# Financial Statement ETL Validation

This notebook reviews data coverage, possible duplicate records, calculated columns, TTM values, one ratio formula, and manually entered comparisons with KAP filings. Internal checks can identify inconsistencies; comparisons with the original filings are needed to assess parsing accuracy.

The final section has blank screenshot slots for KAP statement evidence.

## 1. Scope and database

Edit the symbols and quarter range to match the data you want to review. The default sample is the ten non-financial companies in this project.

In [19]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

SYMBOLS = ["THYAO", "TUPRS", "BIMAS", "ASELS", "EREGL", "SISE", "FROTO", "TCELL", "MGROS", "ENKAI"]
BEGIN_YEAR, BEGIN_QUARTER = 2017, 1
END_YEAR, END_QUARTER = 2026, 2

def has_income_statement(path):
    try:
        with sqlite3.connect(path) as conn:
            return conn.execute(
                "SELECT 1 FROM sqlite_master WHERE type='table' AND name='income_statement'"
            ).fetchone() is not None
    except sqlite3.Error:
        return False

DB_PATH = next(
    (folder / "bist_data.db" for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "bist_data.db").is_file() and has_income_statement(folder / "bist_data.db")),
    None,
)
if DB_PATH is None:
    raise FileNotFoundError("Could not find bist_data.db with an income_statement table from the notebook folder upward.")

print(f"Database: {DB_PATH}")
print(f"Reviewing {len(SYMBOLS)} companies, {BEGIN_YEAR} Q{BEGIN_QUARTER} through {END_YEAR} Q{END_QUARTER}")

Database: /Users/kaanturgut/python/bist-financial-statements-etl/bist_data.db
Reviewing 10 companies, 2017 Q1 through 2026 Q2


## 2. Table inventory and coverage

A missing quarter is a review prompt, not automatically an error. A company may have listed later, lacked a filing, or have an archive that was not downloaded.

In [20]:
from IPython.display import display
import pandas as pd
import sqlite3

tables_to_check = ["income_statement", "balance_sheet", "cash_flow", "equity_change"]

start_period = BEGIN_YEAR * 4 + BEGIN_QUARTER
end_period = END_YEAR * 4 + END_QUARTER
periods = [
    ((period - 1) // 4, (period - 1) % 4 + 1)
    for period in range(start_period, end_period + 1)
]

expected_pairs = {
    (symbol, year, quarter)
    for symbol in SYMBOLS
    for year, quarter in periods
}

symbol_marks = ",".join("?" for _ in SYMBOLS)
summary = []
missing_periods = []

for table in tables_to_check:
    with sqlite3.connect(DB_PATH) as conn:
        rows = conn.execute(
            f"""
            SELECT DISTINCT company, year, quarter
            FROM {table}
            WHERE company IN ({symbol_marks})
              AND (year * 4 + quarter) BETWEEN ? AND ?
            """,
            [*SYMBOLS, start_period, end_period],
        ).fetchall()

    existing_pairs = {(company, year, quarter) for company, year, quarter in rows}
    missing = expected_pairs - existing_pairs

    summary.append({
        "table": table,
        "expected_company_quarters": len(expected_pairs),
        "existing_company_quarters": len(existing_pairs & expected_pairs),
        "missing_company_quarters": len(missing),
        "coverage_pct": round(100 * (len(expected_pairs) - len(missing)) / len(expected_pairs), 1),
    })

    missing_periods.extend(
        {"table": table, "company": company, "year": year, "quarter": quarter}
        for company, year, quarter in sorted(missing)
    )

display(pd.DataFrame(summary))
display(pd.DataFrame(missing_periods))

,table,expected_company_quarters,existing_company_quarters,missing_company_quarters,coverage_pct
0,income_statement,380,380,0,100.0
1,balance_sheet,380,380,0,100.0
2,cash_flow,380,380,0,100.0
3,equity_change,380,380,0,100.0


""


In [25]:
import sqlite3
import pandas as pd
from IPython.display import display

start = BEGIN_YEAR * 4 + BEGIN_QUARTER
end = END_YEAR * 4 + END_QUARTER
periods = [
    ((p - 1) // 4, (p - 1) % 4 + 1)
    for p in range(start, end + 1)
]

expected = {
    (symbol, year, quarter)
    for symbol in SYMBOLS
    for year, quarter in periods
}

marks = ",".join("?" for _ in SYMBOLS)
with sqlite3.connect(DB_PATH) as conn:
    found_rows = conn.execute(
        f"""
        SELECT DISTINCT company, year, quarter
        FROM income_statement
        WHERE company IN ({marks})
          AND (year * 4 + quarter) BETWEEN ? AND ?
        """,
        [*SYMBOLS, start, end],
    ).fetchall()

found = set(found_rows)
missing = sorted(expected - found)

missing_df = pd.DataFrame(missing, columns=["company", "year", "quarter"])
display(missing_df)
display(
    missing_df.groupby("company")
    .size()
    .rename("missing_quarters")
    .reset_index()
)

,company,year,quarter


,company,missing_quarters


## 3. Duplicate candidates

The ETL removes exact duplicate rows and selects a canonical filing. These checks flag repeated company-period-section-item labels for review. Repeated labels can sometimes be legitimate, so inspect the source details before treating a result as an error.

In [26]:
duplicate_groups = []

for table in ["income_statement", "balance_sheet", "cash_flow", "equity_change"]:
    with sqlite3.connect(DB_PATH) as conn:
        columns = {row[1] for row in conn.execute(f'PRAGMA table_info({table})')}
    if "line_item_path" in columns:
        line_key = "COALESCE(line_item_path, COALESCE(section, '') || ' > ' || item)"
    elif "section" in columns:
        line_key = "COALESCE(section, '') || ' > ' || item"
    else:
        line_key = "item"
    if {"source_column_id", "column_label"}.issubset(columns):
        column_select = "source_column_id, column_label"
        column_group = ", source_column_id, column_label"
    else:
        column_select = "NULL AS source_column_id, NULL AS column_label"
        column_group = ""
    with sqlite3.connect(DB_PATH) as conn:
        frame = pd.read_sql_query(
            f"""
            SELECT company, year, quarter, date, section, {line_key} AS line_item_path, item,
                   {column_select},
                   COUNT(*) AS row_count,
                   GROUP_CONCAT(source_filename) AS source_files
            FROM {table}
            WHERE company IN ({symbol_marks})
              AND value != 0
            GROUP BY company, year, quarter, date, {line_key}, item{column_group}
            HAVING COUNT(*) > 1
            ORDER BY row_count DESC, company, year, quarter
            """,
            conn,
            params=SYMBOLS,
        )

    if not frame.empty:
        frame.insert(0, "table", table)
        duplicate_groups.append(frame)

duplicate_groups = (
    pd.concat(duplicate_groups, ignore_index=True)
    if duplicate_groups else pd.DataFrame()
)

print(f"Candidate duplicate groups excluding zero values: {len(duplicate_groups)}")
display(duplicate_groups.head(100))

Candidate duplicate groups excluding zero values: 0


""


## 4. Inflation and USD calculations

The pipeline calculates value_adj_inflation as value multiplied by inflation_multiplier, and value_usd as value divided by usd_try. This recomputes both from the stored inputs and reports mismatches and missing reference values.

In [27]:
def check_adjustments(table):
    with sqlite3.connect(DB_PATH) as conn:
        frame = pd.read_sql_query(
            f"SELECT company, year, quarter, item, value, value_adj_inflation, "
            f"inflation_multiplier, value_usd, usd_try FROM {table} "
            f"WHERE company IN ({symbol_marks})",
            conn,
            params=SYMBOLS,
        )
    frame["expected_inflation_value"] = frame["value"] * frame["inflation_multiplier"]
    frame["expected_usd_value"] = frame["value"] / frame["usd_try"]
    inflation_ok = np.isclose(
        frame["value_adj_inflation"], frame["expected_inflation_value"],
        rtol=1e-9, atol=1e-6, equal_nan=True,
    )
    usd_ok = np.isclose(
        frame["value_usd"], frame["expected_usd_value"],
        rtol=1e-9, atol=1e-6, equal_nan=True,
    )
    summary = {
        "table": table,
        "rows": len(frame),
        "missing_inflation_multiplier": int(frame["inflation_multiplier"].isna().sum()),
        "inflation_mismatches": int((~inflation_ok).sum()),
        "missing_usd_rate": int(frame["usd_try"].isna().sum()),
        "usd_mismatches": int((~usd_ok).sum()),
    }
    return summary, frame.loc[~(inflation_ok & usd_ok)].copy()

summaries, mismatch_frames = [], []
for table in ["income_statement", "balance_sheet", "cash_flow", "equity_change"]:
    summary, mismatches = check_adjustments(table)
    summaries.append(summary)
    if not mismatches.empty:
        mismatches.insert(0, "table", table)
        mismatch_frames.append(mismatches)

display(pd.DataFrame(summaries))
adjustment_mismatches = pd.concat(mismatch_frames, ignore_index=True) if mismatch_frames else pd.DataFrame()
display(adjustment_mismatches.head(30))

,table,rows,missing_inflation_multiplier,inflation_mismatches,missing_usd_rate,usd_mismatches
0,income_statement,15393,0,0,0,0
1,balance_sheet,31688,0,0,0,0
2,cash_flow,21815,0,0,0,0
3,equity_change,86586,0,0,0,0


""


## 5. TTM validation

For income statement items, TTM is the sum of the current quarter and previous three quarter-periods when four records exist. This query independently recomputes the nominal and inflation-adjusted sums.

In [28]:
line_path_t = """
COALESCE(t.line_item_path, COALESCE(t.section, '') || ' > ' || t.item)
"""
line_path_p = """
COALESCE(p.line_item_path, COALESCE(p.section, '') || ' > ' || p.item)
"""
line_path_i = """
COALESCE(i.line_item_path, COALESCE(i.section, '') || ' > ' || i.item)
"""

with sqlite3.connect(DB_PATH) as conn:
    ttm_check = pd.read_sql_query(
        f"""
        WITH expected AS (
            SELECT
                t.company,
                t.year,
                t.quarter,
                {line_path_t} AS line_item_path,
                t.item,
                COUNT(p.value) AS period_count,
                SUM(p.value) AS expected_ttm_value,
                SUM(p.value_adj_inflation) AS expected_ttm_real
            FROM income_statement AS t
            LEFT JOIN income_statement AS p
                ON p.company = t.company
                AND {line_path_p} = {line_path_t}
                AND (p.year * 4 + p.quarter)
                    BETWEEN (t.year * 4 + t.quarter - 3)
                    AND (t.year * 4 + t.quarter)
            WHERE t.company IN ({symbol_marks})
            GROUP BY
                t.company,
                t.year,
                t.quarter,
                {line_path_t},
                t.item
        )
        SELECT
            i.company,
            i.year,
            i.quarter,
            e.line_item_path,
            i.item,
            e.period_count,
            i.ttm_value,
            e.expected_ttm_value,
            i.ttm_value_adj_inflation,
            e.expected_ttm_real
        FROM income_statement AS i
        JOIN expected AS e
            ON i.company = e.company
            AND i.year = e.year
            AND i.quarter = e.quarter
            AND {line_path_i} = e.line_item_path
            AND i.item = e.item
        WHERE e.period_count = 4
        ORDER BY i.company, i.year, i.quarter, e.line_item_path
        """,
        conn,
        params=SYMBOLS,
    )

numeric_columns = [
    "ttm_value",
    "expected_ttm_value",
    "ttm_value_adj_inflation",
    "expected_ttm_real",
]

for column in numeric_columns:
    ttm_check[column] = pd.to_numeric(ttm_check[column], errors="coerce")

ttm_check["nominal_matches"] = np.isclose(
    ttm_check["ttm_value"],
    ttm_check["expected_ttm_value"],
    rtol=1e-9,
    atol=1e-6,
    equal_nan=True,
)
ttm_check["real_matches"] = np.isclose(
    ttm_check["ttm_value_adj_inflation"],
    ttm_check["expected_ttm_real"],
    rtol=1e-9,
    atol=1e-6,
    equal_nan=True,
)

mismatch_mask = ~(ttm_check["nominal_matches"] & ttm_check["real_matches"])

print(
    f"Checked {len(ttm_check)} rows; "
    f"nominal mismatches: {(~ttm_check['nominal_matches']).sum()}; "
    f"inflation-adjusted mismatches: {(~ttm_check['real_matches']).sum()}"
)
display(ttm_check.loc[mismatch_mask].head(30))

Checked 13384 rows; nominal mismatches: 0; inflation-adjusted mismatches: 0


,company,year,quarter,line_item_path,item,period_count,ttm_value,expected_ttm_value,ttm_value_adj_inflation,expected_ttm_real,nominal_matches,real_matches


## 6. Ratio spot check: quarterly gross margin

This independently recalculates gross margin from statement rows and compares it with the stored ratio. It checks one ratio formula and its data path, not every ratio definition.

In [29]:
with sqlite3.connect(DB_PATH) as conn:
    source = pd.read_sql_query(
        f"SELECT company, year, quarter, item, value FROM income_statement "
        f"WHERE company IN ({symbol_marks}) AND item IN ('hasılat', 'satışların maliyeti')",
        conn,
        params=SYMBOLS,
    )
    stored = pd.read_sql_query(
        f"SELECT company, year, quarter, gross_margin_qrt FROM ratios WHERE company IN ({symbol_marks})",
        conn,
        params=SYMBOLS,
    )

wide = source.groupby(["company", "year", "quarter", "item"])["value"].sum().unstack("item").reset_index()
wide["expected_gross_margin_qrt"] = (
    wide["hasılat"] - wide["satışların maliyeti"].abs()
) / wide["hasılat"]
gross_margin_check = wide.merge(stored, on=["company", "year", "quarter"], how="left")
gross_margin_check["matches"] = np.isclose(
    gross_margin_check["gross_margin_qrt"],
    gross_margin_check["expected_gross_margin_qrt"],
    rtol=1e-8,
    atol=1e-8,
    equal_nan=True,
)
print(f"Gross-margin rows checked: {len(gross_margin_check)}; mismatches: {(~gross_margin_check['matches']).sum()}")
display(gross_margin_check.loc[~gross_margin_check["matches"]].head(30))

Gross-margin rows checked: 380; mismatches: 0


,company,year,quarter,hasılat,satışların maliyeti,expected_gross_margin_qrt,gross_margin_qrt,matches


## 7. Compare with a KAP filing

For income statements, KAP may report year-to-date cumulative values while this database stores standalone quarters. Enter the filing's cumulative figure and the prior cumulative figure; for Q1 use zero, and for Q4 use the nine-month cumulative figure as the prior value. The notebook subtracts them before comparing. Keep units consistent and record the filing link or ID.

In [30]:
# Replace the example with values transcribed from the original KAP statement.
manual_source_values = pd.DataFrame([
    {
        "company": "THYAO",
        "year": 2025,
        "quarter": 4,
        "item": "hasılat",
        "kap_cumulative_value": 955472000000,
        "prior_cumulative_value": 690825000000,
        "source_note": "Add KAP filing URL or report ID",
    },
])

with sqlite3.connect(DB_PATH) as conn:
    database_values = pd.read_sql_query(
        f"SELECT company, year, quarter, item, value, value_adj_inflation, value_usd, "
        f"report_kind, filing_id, source_filename, file_path FROM income_statement "
        f"WHERE company IN ({symbol_marks})",
        conn,
        params=SYMBOLS,
    )

source_comparison = manual_source_values.merge(
    database_values, on=["company", "year", "quarter", "item"], how="left"
)
source_comparison["kap_standalone_quarter_value"] = source_comparison["kap_cumulative_value"] - source_comparison["prior_cumulative_value"]
source_comparison["difference"] = source_comparison["value"] - source_comparison["kap_standalone_quarter_value"]
source_comparison["absolute_difference"] = source_comparison["difference"].abs()
display(source_comparison)

,company,year,quarter,item,kap_cumulative_value,prior_cumulative_value,source_note,value,value_adj_inflation,value_usd,report_kind,filing_id,source_filename,file_path,kap_standalone_quarter_value,difference,absolute_difference
0,THYAO,2025,4,hasılat,955472000000,690825000000,Add KAP filing URL or report ID,2.646470e+11,3.289522e+11,6.233295e+09,Konsolide,1565996,THYAO_1565996_2025_4.xls,/Users/kaanturgut/python/bist-financial-statem...,264647000000,0.0,0.0


## 8. Visual review

Use the plots to spot unusual jumps or flat series. A change may be a real business event, reporting revision, or parsing issue; check the filing before drawing a conclusion.

In [31]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import qualitative

PLOT_SYMBOL = "MGROS"

with sqlite3.connect(DB_PATH) as conn:
    plotted = pd.read_sql_query(
        """
        SELECT year, quarter, item, value, value_adj_inflation
        FROM income_statement
        WHERE company = ?
          AND item IN ('hasılat', 'dönem karı (zararı)')
        ORDER BY year, quarter
        """,
        conn,
        params=(PLOT_SYMBOL,),
    )

plotted["period"] = (
    plotted["year"].astype(str) + " Q" + plotted["quarter"].astype(str)
)
period_order = plotted["period"].drop_duplicates().tolist()

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=(
        f"{PLOT_SYMBOL}: nominal quarterly values",
        f"{PLOT_SYMBOL}: inflation-adjusted quarterly values",
    ),
)

colors = qualitative.Plotly

for i, (item, group) in enumerate(plotted.groupby("item", sort=True)):
    color = colors[i % len(colors)]

    fig.add_trace(
        go.Scatter(
            x=group["period"],
            y=group["value"] / 1e9,
            mode="lines+markers",
            name=item,
            legendgroup=item,
            line={"color": color},
            marker={"color": color, "size": 7},
            hovertemplate=f"%{{x}}<br>%{{y:.2f}} TRY bn<extra>{item}</extra>",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=group["period"],
            y=group["value_adj_inflation"] / 1e9,
            mode="lines+markers",
            name=item,
            legendgroup=item,
            showlegend=False,
            line={"color": color},
            marker={"color": color, "size": 7},
            hovertemplate=f"%{{x}}<br>%{{y:.2f}} TRY bn<extra>{item}</extra>",
        ),
        row=2,
        col=1,
    )

fig.update_xaxes(
    categoryorder="array",
    categoryarray=period_order,
    tickangle=70,
    showgrid=True,
    gridcolor="rgba(128,128,128,0.25)",
    row=2,
    col=1,
)
fig.update_yaxes(title_text="TRY billions", showgrid=True, row=1, col=1)
fig.update_yaxes(title_text="TRY billions", showgrid=True, row=2, col=1)

fig.update_layout(
    height=800,
    width=1300,
    template="plotly_white",
    legend_title_text="Item",
    hovermode="x unified",
)

fig.show()